In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_IHBAS_Dilshad_Garden_Delhi_CPCB_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,326.0,113.0,133.0,133.0,140.0,225.0,59.0,NaN,79.0,118.0,242.0,281.0
1,2,338.0,115.0,95.0,131.0,130.0,168.0,NaN,73.0,79.0,121.0,295.0,263.0
2,3,276.0,112.0,87.0,169.0,148.0,147.0,93.0,65.0,76.0,122.0,275.0,263.0
3,4,347.0,145.0,128.0,177.0,161.0,186.0,61.0,66.0,77.0,142.0,326.0,196.0
4,5,321.0,96.0,NaN,145.0,170.0,222.0,55.0,53.0,66.0,123.0,282.0,203.0
5,6,284.0,99.0,94.0,145.0,173.0,155.0,51.0,64.0,83.0,112.0,261.0,237.0
6,7,215.0,121.0,123.0,152.0,199.0,180.0,45.0,67.0,74.0,110.0,264.0,198.0
7,8,229.0,NaN,119.0,145.0,161.0,190.0,48.0,65.0,82.0,178.0,265.0,250.0
8,9,257.0,NaN,101.0,192.0,131.0,171.0,73.0,69.0,92.0,169.0,216.0,163.0
9,10,120.0,189.0,NaN,NaN,118.0,152.0,108.0,85.0,81.0,152.0,262.0,165.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,326.000000,113.000000,133.000000,133.000000,140.000000,225.000000,59.000000,67.586207,79.00000,118.000000,242.000000,281.000000
1,2,338.000000,115.000000,95.000000,131.000000,130.000000,168.000000,70.542857,73.000000,79.00000,121.000000,295.000000,263.000000
2,3,276.000000,112.000000,87.000000,169.000000,148.000000,147.000000,93.000000,65.000000,76.00000,122.000000,275.000000,263.000000
3,4,347.000000,145.000000,128.000000,177.000000,161.000000,186.000000,61.000000,66.000000,77.00000,142.000000,326.000000,196.000000
4,5,321.000000,96.000000,111.733333,145.000000,170.000000,222.000000,55.000000,67.586207,66.00000,123.000000,282.000000,203.000000
5,6,284.000000,99.000000,94.000000,145.000000,173.000000,155.000000,51.000000,64.000000,83.00000,112.000000,261.000000,237.000000
6,7,215.000000,121.000000,123.000000,152.000000,199.000000,180.000000,45.000000,67.000000,74.00000,110.000000,264.000000,198.000000
7,8,229.000000,131.032258,119.000000,145.000000,161.000000,190.000000,48.000000,65.000000,82.00000,178.000000,265.000000,250.000000
8,9,257.000000,131.032258,101.000000,138.451613,131.000000,171.000000,73.000000,69.000000,92.00000,169.000000,216.000000,163.000000
9,10,248.542857,189.000000,111.733333,138.451613,118.000000,152.000000,108.000000,67.586207,81.00000,152.000000,262.000000,165.000000
